# Lesson 9.2 — Database Migration: AWS DMS & Schema Conversion Tool

**Level: Advanced** | **Exam:** SAP-C02 Domain 4, Task 4.2

## Database Migration Tools

### AWS Database Migration Service (DMS)
Migrates databases to AWS with minimal downtime:
- **Full load**: Copy all existing data
- **CDC (Change Data Capture)**: Continuously replicate ongoing changes
- **Full load + CDC**: Initial copy + keep in sync → minimize cutover window

```
Source DB ─→ DMS Replication Instance ─→ Target DB
            (runs on EC2, you choose size)
```

### Source → Target Combinations (DMS supports heterogeneous migrations)
```
Oracle ─→ Aurora PostgreSQL    (change DB engine)
SQL Server ─→ Aurora MySQL     (change DB engine)
MySQL ─→ Aurora MySQL          (same engine, cloud version)
Oracle ─→ Oracle RDS           (lift-and-shift)
```

### AWS Schema Conversion Tool (SCT)
- Converts database **schema and code** for heterogeneous migrations
- Automatically converts: tables, views, stored procedures, functions, triggers
- Marks items it cannot convert (manual review required)
- Works offline (desktop application)

### DMS Task Types
| Task | Description |
|------|------------|
| **Full load** | One-time migration, downtime required |
| **CDC only** | Ongoing replication (must have existing schema) |
| **Full load + CDC** | Best practice: copy data, then switch to CDC, then cut over |

### Zero-Downtime Migration Pattern
```
1. Create DMS task: Full load + CDC
2. Wait for full load to complete
3. CDC keeps target in sync with source
4. Test target database thoroughly
5. Update application to use target DB endpoint
6. Stop DMS task
7. Decommission source DB
```

In [ ]:
import boto3, json
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# DMS inventory and migration planning
import boto3, json

dms = boto3.client('dms', region_name='us-east-1')

# List DMS replication instances
try:
    instances = dms.describe_replication_instances()['ReplicationInstances']
    print(f"DMS Replication Instances ({len(instances)}):")
    for inst in instances:
        print(f"  {inst['ReplicationInstanceIdentifier']:30} {inst['ReplicationInstanceClass']:15} {inst['ReplicationInstanceStatus']}")
        
    # List replication tasks
    tasks = dms.describe_replication_tasks()['ReplicationTasks']
    print(f"\nDMS Replication Tasks ({len(tasks)}):")
    for task in tasks:
        print(f"  {task['ReplicationTaskIdentifier']:30} {task['MigrationType']:20} {task['Status']}")
        
    if not instances:
        print("  No DMS instances found.")
        
except Exception as e:
    print(f"DMS: {e}")

# Database migration decision matrix
print("\n=== Heterogeneous DB Migration Decision ===")
migrations = [
    ("Oracle → Aurora PostgreSQL", "SCT (schema convert) + DMS (data migrate)", "High complexity"),
    ("MySQL → Aurora MySQL", "DMS only (no schema conversion needed)", "Low complexity"),
    ("SQL Server → RDS SQL Server", "DMS only (same engine)", "Low complexity"),
    ("Oracle → DynamoDB", "DMS + manual data model redesign", "Very high complexity"),
    ("MongoDB → DocumentDB", "Mongodump/restore or DMS", "Medium complexity"),
]
for source_target, tool, complexity in migrations:
    print(f"  {source_target:40} Tool: {tool:45} {complexity}")

## Exercise

In [ ]:
# TODO: Plan a database migration from Oracle to Aurora PostgreSQL
# Fill in each step of the migration plan with the appropriate action

migration_plan = {
    "phase_1_assessment": {
        "tool": "TODO",           # Which tool to assess schema compatibility?
        "action": "TODO",          # What does this tool do?
        "output": "TODO"           # What is produced?
    },
    "phase_2_schema_conversion": {
        "tool": "TODO",
        "action": "TODO",
        "handle_unconverted": "TODO"  # How to handle items SCT can't convert?
    },
    "phase_3_initial_load": {
        "tool": "TODO",
        "task_type": "TODO",       # Full load / CDC / Full load + CDC
        "downtime_required": "TODO"
    },
    "phase_4_cdc_sync": {
        "tool": "TODO",
        "purpose": "TODO",
        "monitoring": "TODO"       # How to monitor replication lag?
    },
    "phase_5_cutover": {
        "steps": [
            "TODO",    # Step 1
            "TODO",    # Step 2
            "TODO",    # Step 3
            "TODO"     # Step 4
        ]
    }
}

import json
print(json.dumps(migration_plan, indent=2))

## Summary
- **DMS**: Migrates data with minimal downtime; supports homogeneous and heterogeneous migrations
- **SCT**: Converts schema/code for heterogeneous migrations (Oracle→PostgreSQL, SQL Server→MySQL)
- **Full load + CDC**: Best practice — copy initial data, continuously sync changes, then cut over
- **DMS replication instance** size matters — too small = migration backlog; monitor replication lag
- After migration: update application connection strings (ideally via environment variables or Secrets Manager)

---
**Next:** [Lesson 9.3 — Application Migration: AWS MGN, DataSync & Snow Family](lesson-9.3-application-migration.ipynb)